# RAG Experiments (Staged Evaluation)

This notebook is dedicated to reproducible experiments.

- Stage A: ingestion (`ingestion_seconds`)
- Stage B: retrieval-only (`retrieval_only_seconds`)
- Stage C: answer generation from retrieved docs (`generation_only_seconds`)
- Query-to-response metric: `query_to_response_seconds = Stage B + Stage C`

It also exports retrieved chunks for manual audit. RAGAS evaluation is commented out until there are benchmark questions and reference answers.

In [32]:
import csv
import hashlib
import json
import os
import random
import time
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Literal

import chromadb
from IPython.display import Markdown, display
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_community.document_loaders import DirectoryLoader, UnstructuredPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

In [33]:
@dataclass
class ExperimentConfig:
    embedding_model: Literal["nomic-embed-text", "bge-m3"] = "nomic-embed-text"
    chunking_strategy: Literal["fixed", "recursive", "semantic"] = "semantic"
    chunk_params: Dict[str, Any] = None
    retriever_params: Dict[str, Any] = None

    llm_model: str = "llama3.2"
    llm_temperature: float = 0.1

    data_dir: str = "data_folder/"
    chroma_path: str = "chroma_database"
    collection_prefix: str = "rag_chatbot"

    logs_dir: str = "runs"
    random_seed: int = 42


def _defaults_if_missing(config: ExperimentConfig) -> ExperimentConfig:
    if config.chunk_params is None:
        config.chunk_params = {
            "chunk_size": 1000,
            "chunk_overlap": 200,
            "breakpoint_threshold_type": "percentile",
            "separator": "\n\n",
        }
    if config.retriever_params is None:
        config.retriever_params = {
            "k": 4,
            "fetch_k": 20,
            "lambda_mult": 0.6,
            "search_type": "mmr",
        }
    return config


def set_reproducibility(seed: int = 42) -> None:
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except Exception:
        pass


def _safe_collection_name(config: ExperimentConfig) -> str:
    # Separate collections by embedding/chunker to avoid embedding-dimension collisions.
    return f"{config.collection_prefix}_{config.embedding_model}_{config.chunking_strategy}".replace("-", "_")


def build_embeddings(config: ExperimentConfig):
    model = config.embedding_model.lower()

    if model == "nomic-embed-text":
        return OllamaEmbeddings(model="nomic-embed-text")

    if model == "bge-m3":
        try:
            from langchain_huggingface import HuggingFaceEmbeddings
        except Exception as e:
            raise RuntimeError(
                "BGE-M3 requires HuggingFace support. Install: pip install langchain-huggingface sentence-transformers"
            ) from e

        return HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

    raise ValueError(f"Unsupported embedding model: {config.embedding_model}")


def build_chunker(config: ExperimentConfig, embeddings):
    strategy = config.chunking_strategy.lower()
    p = config.chunk_params

    if strategy == "fixed":
        return CharacterTextSplitter(
            chunk_size=int(p.get("chunk_size", 1000)),
            chunk_overlap=int(p.get("chunk_overlap", 200)),
            separator=p.get("separator", "\n\n"),
        )

    if strategy == "recursive":
        return RecursiveCharacterTextSplitter(
            chunk_size=int(p.get("chunk_size", 1000)),
            chunk_overlap=int(p.get("chunk_overlap", 200)),
        )

    if strategy == "semantic":
        return SemanticChunker(
            embeddings,
            breakpoint_threshold_type=p.get("breakpoint_threshold_type", "percentile"),
        )

    raise ValueError(f"Unsupported chunking strategy: {config.chunking_strategy}")


def load_documents(data_dir: str):
    loader = DirectoryLoader(
        data_dir,
        glob="**/*.pdf",
        loader_cls=UnstructuredPDFLoader,
        show_progress=True,
    )
    documents = loader.load()
    if not documents:
        raise RuntimeError(f"No PDF files found in '{data_dir}'.")

    for doc in documents:
        source_path = doc.metadata.get("source", "")
        if source_path and os.path.exists(source_path):
            doc.metadata["file_path"] = source_path
            with open(source_path, "rb") as f:
                doc.metadata["file_hash"] = hashlib.md5(f.read()).hexdigest()

    return documents


def split_documents(documents, chunker):
    return chunker.split_documents(documents)


def build_vector_db(chunks, embeddings, config: ExperimentConfig):
    client = chromadb.PersistentClient(path=config.chroma_path)
    collection_name = _safe_collection_name(config)

    try:
        client.get_collection(name=collection_name)
        vector_db = Chroma(
            collection_name=collection_name,
            embedding_function=embeddings,
            client=client,
        )
        vector_db.delete_collection()
    except Exception:
        pass

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name,
        client=client,
    )


def build_retriever(vector_db, llm, config: ExperimentConfig):
    p = config.retriever_params
    base_retriever = vector_db.as_retriever(
        search_type=p.get("search_type", "mmr"),
        search_kwargs={
            "k": int(p.get("k", 4)),
            "fetch_k": int(p.get("fetch_k", 20)),
            "lambda_mult": float(p.get("lambda_mult", 0.6)),
        },
    )

    query_prompt = PromptTemplate(
        input_variables=["question"],
        template="""You are a query rephrasing assistant for vector search.
Generate 2 alternative, semantically diverse versions of the user's question.
Return only the rephrased questions, one per line.
Original question: {question}""",
    )

    return MultiQueryRetriever.from_llm(base_retriever, llm, prompt=query_prompt)


def format_docs(docs):
    rows = []
    for doc in docs:
        src = os.path.basename(doc.metadata.get("source", "Unknown"))
        rows.append(f"Document Source: {src}\nContent: {doc.page_content}")
    return "\n\n---\n\n".join(rows)


def generate_answer_from_docs(question: str, docs, config: ExperimentConfig) -> str:
    llm = ChatOllama(model=config.llm_model, temperature=config.llm_temperature)
    template = """You are a helpful and accurate assistant. You answer in the same language as the question.

Answer the question using ONLY the provided Context information below.
If context is insufficient, explicitly say the information is not available.

Context: {context}
Question: {question}

Answer:
"""
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": format_docs(docs), "question": question})


def _approx_token_count(text: str) -> int:
    return max(1, int(len(text) / 4))


def _chunk_stats(chunks):
    if not chunks:
        return {"chunk_count": 0, "avg_chunk_chars": 0.0, "avg_chunk_tokens_approx": 0.0}

    lengths = [len(c.page_content) for c in chunks]
    token_estimates = [_approx_token_count(c.page_content) for c in chunks]
    return {
        "chunk_count": len(chunks),
        "avg_chunk_chars": sum(lengths) / len(lengths),
        "avg_chunk_tokens_approx": sum(token_estimates) / len(token_estimates),
    }

# this value is stored in the run metrics (and CSV) so one can tell later which corpus a logged experiment used, without storing full paths in every row or re-reading all PDFs
def _dataset_fingerprint(documents):
    parts = []
    for d in documents:
        p = d.metadata.get("file_path", "")
        h = d.metadata.get("file_hash", "")
        parts.append(f"{p}:{h}")
    joined = "|".join(sorted(parts))
    return hashlib.md5(joined.encode("utf-8")).hexdigest()

HIT RATE & LOGGING

In [34]:
def export_retrieved_chunks_for_audit(run_id: str, question: str, docs, logs_dir: str):
    logs = Path(logs_dir)
    logs.mkdir(parents=True, exist_ok=True)

    payload = {
        "run_id": run_id,
        "question": question,
        "retrieved_k": len(docs),
        "chunks": [
            {
                "rank": i + 1,
                "source": d.metadata.get("source", ""),
                "file_path": d.metadata.get("file_path", ""),
                "file_hash": d.metadata.get("file_hash", ""),
                "content": d.page_content,
            }
            for i, d in enumerate(docs)
        ],
    }

    json_path = logs / f"{run_id}_retrieved_chunks.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    csv_path = logs / "retrieved_chunks_audit.csv"
    rows = []
    for c in payload["chunks"]:
        rows.append(
            {
                "run_id": run_id,
                "question": question,
                "rank": c["rank"],
                "source": c["source"],
                "file_path": c["file_path"],
                "file_hash": c["file_hash"],
                "content": c["content"],
            }
        )

    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["run_id", "question", "rank", "source", "file_path", "file_hash", "content"])
        if write_header:
            writer.writeheader()
        if rows:
            writer.writerows(rows)

    return str(json_path), str(csv_path)


def log_experiment_row(config: ExperimentConfig, metrics: Dict[str, Any]):
    logs_dir = Path(config.logs_dir)
    logs_dir.mkdir(parents=True, exist_ok=True)

    run_id = metrics["run_id"]
    json_path = logs_dir / f"{run_id}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    csv_path = logs_dir / "metrics.csv"
    row = {
        "run_id": run_id,
        "timestamp_utc": metrics["timestamp_utc"],
        "embedding_model": config.embedding_model,
        "chunking_strategy": config.chunking_strategy,
        "ingestion_seconds": metrics["ingestion_seconds"],
        "retrieval_only_seconds": metrics["retrieval_only_seconds"],
        "generation_only_seconds": metrics["generation_only_seconds"],
        "query_to_response_seconds": metrics["query_to_response_seconds"],
        "end_to_end_seconds": metrics["end_to_end_seconds"],
        "chunk_count": metrics["chunk_count"],
        "avg_chunk_chars": metrics["avg_chunk_chars"],
        "avg_chunk_tokens_approx": metrics["avg_chunk_tokens_approx"],
        "k": config.retriever_params.get("k", 4),
        "fetch_k": config.retriever_params.get("fetch_k", 20),
        "dataset_fingerprint": metrics["dataset_fingerprint"],
    }

    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    return str(json_path), str(csv_path)


def run_experiment_staged(config: ExperimentConfig, question: str):
    config = _defaults_if_missing(config)
    set_reproducibility(config.random_seed)

    run_id = uuid.uuid4().hex[:12]
    ts = datetime.now(timezone.utc).isoformat()
    t0 = time.perf_counter()

    embeddings = build_embeddings(config)
    chunker = build_chunker(config, embeddings)

    # Stage A: ingestion (load + chunk + index)
    t_a = time.perf_counter()
    documents = load_documents(config.data_dir)
    chunks = split_documents(documents, chunker)
    vector_db = build_vector_db(chunks, embeddings, config)
    ingestion_seconds = time.perf_counter() - t_a

    retriever_llm = ChatOllama(model=config.llm_model, temperature=config.llm_temperature)
    retriever = build_retriever(vector_db, retriever_llm, config)

    # Query-to-response window starts here.
    t_q = time.perf_counter()

    # Stage B: retrieval only
    t_b = time.perf_counter()
    retrieved_docs = retriever.invoke(question)
    retrieval_only_seconds = time.perf_counter() - t_b

    # Stage C: generation only (on already retrieved docs)
    t_c = time.perf_counter()
    answer = generate_answer_from_docs(question, retrieved_docs, config)
    generation_only_seconds = time.perf_counter() - t_c

    query_to_response_seconds = time.perf_counter() - t_q
    end_to_end_seconds = time.perf_counter() - t0

    stats = _chunk_stats(chunks)
    metrics = {
        "run_id": run_id,
        "timestamp_utc": ts,
        "config": asdict(config),
        "ingestion_seconds": round(ingestion_seconds, 4),
        "retrieval_only_seconds": round(retrieval_only_seconds, 4),
        "generation_only_seconds": round(generation_only_seconds, 4),
        "query_to_response_seconds": round(query_to_response_seconds, 4),
        "end_to_end_seconds": round(end_to_end_seconds, 4),
        "chunk_count": stats["chunk_count"],
        "avg_chunk_chars": round(stats["avg_chunk_chars"], 2),
        "avg_chunk_tokens_approx": round(stats["avg_chunk_tokens_approx"], 2),
        "dataset_fingerprint": _dataset_fingerprint(documents),
    }

    run_json_path, run_csv_path = log_experiment_row(config, metrics)
    chunks_json_path, chunks_csv_path = export_retrieved_chunks_for_audit(
        run_id=run_id,
        question=question,
        docs=retrieved_docs,
        logs_dir=config.logs_dir,
    )

    return {
        "answer": answer,
        "retrieved_docs": retrieved_docs,
        "metrics": metrics,
        "run_json": run_json_path,
        "metrics_csv": run_csv_path,
        "chunks_json": chunks_json_path,
        "chunks_csv": chunks_csv_path,
    }


def display_retrieved_chunks(docs, max_chars: int = 600):
    for i, d in enumerate(docs, start=1):
        src = os.path.basename(d.metadata.get("source", "Unknown"))
        snippet = d.page_content[:max_chars]
        print(f"[{i}] source={src}")
        print(snippet)
        print("-" * 80)


# RAGAS: uncomment when you have benchmark Q&A and `pip install ragas datasets`
# def evaluate_with_ragas_scaffold(records: List[Dict[str, Any]]):
#     """Scaffold: records should include question, answer, contexts, and optional ground_truth."""
#     try:
#         from datasets import Dataset
#         from ragas import evaluate
#         from ragas.metrics import answer_relevancy, faithfulness
#     except Exception as e:
#         print("RAGAS not installed yet. Install with: pip install ragas datasets")
#         print(f"Import error: {e}")
#         return None
#
#     dataset = Dataset.from_list(records)
#     result = evaluate(dataset=dataset, metrics=[faithfulness, answer_relevancy])
#     return result

In [35]:
# Example run
config = ExperimentConfig(
    embedding_model="nomic-embed-text",  # or "bge-m3" | "nomic-embed-text"
    chunking_strategy="semantic",         # "fixed" | "recursive" | "semantic"
    chunk_params={
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "breakpoint_threshold_type": "percentile",
    },
    retriever_params={"k": 4, "fetch_k": 20, "lambda_mult": 0.6, "search_type": "mmr"},
)

result = run_experiment_staged(config, "What are these documents about?")
print("Run ID:", result["metrics"]["run_id"])
print("Metrics:", json.dumps(result["metrics"], indent=2))
print("Run JSON:", result["run_json"])
print("Metrics CSV:", result["metrics_csv"])
print("Retrieved Chunks JSON:", result["chunks_json"])
print("Retrieved Chunks CSV:", result["chunks_csv"])

display_retrieved_chunks(result["retrieved_docs"], max_chars=450)
display(Markdown(result["answer"]))

  0%|          | 0/1 [00:00<?, ?it/s]Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
No languages specified, defaulting to English.
100%|██████████| 1/1 [00:00<00:00,  3.31it/s]


Run ID: f5fd83c75dd2
Metrics: {
  "run_id": "f5fd83c75dd2",
  "timestamp_utc": "2026-04-05T10:20:07.346444+00:00",
  "config": {
    "embedding_model": "nomic-embed-text",
    "chunking_strategy": "semantic",
    "chunk_params": {
      "chunk_size": 1000,
      "chunk_overlap": 200,
      "breakpoint_threshold_type": "percentile"
    },
    "retriever_params": {
      "k": 4,
      "fetch_k": 20,
      "lambda_mult": 0.6,
      "search_type": "mmr"
    },
    "llm_model": "llama3.2",
    "llm_temperature": 0.1,
    "data_dir": "data_folder/",
    "chroma_path": "chroma_database",
    "collection_prefix": "rag_chatbot",
    "logs_dir": "runs",
    "random_seed": 42
  },
  "ingestion_seconds": 1.4196,
  "retrieval_only_seconds": 2.2489,
  "generation_only_seconds": 4.4758,
  "query_to_response_seconds": 6.7247,
  "end_to_end_seconds": 8.1852,
  "chunk_count": 3,
  "avg_chunk_chars": 1662.33,
  "avg_chunk_tokens_approx": 415.33,
  "dataset_fingerprint": "ad558587634c500e7e6d58cc00b70f31"

Diese Dokumente sind über die Technischen Rahmenbedingungen (Technische Anforderungen) für ein Task-Management-System namens TaskFlow. Sie enthalten spezifische Anforderungen an das System, wie z.B. die Implementierung einer REST-API, die Datenpersistierung in einer SQLite-Datenbank und die Integration von Benachrichtigungen.

In [36]:
# RAGAS scaffold (uncomment when you have benchmark Q&A and the helper above is uncommented)
# ragas_records = [
#     {
#         "question": "What are these documents about?",
#         "answer": result["answer"],
#         "contexts": [d.page_content for d in result["retrieved_docs"]],
#         "ground_truth": "",  # add reference answer when available
#     }
# ]
# ragas_result = evaluate_with_ragas_scaffold(ragas_records)
# ragas_result